In [3]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import deque
from ultralytics import YOLO

# 1. 에러 메시지 완벽 분석 기반 모델 재구성
# 파일 가중치 이름: "attention.0...", "classifier.1, 4, 7..."
# LSTM 입력 차원: [512, 85] -> input_size=85, hidden_size=128
class FinalCorrectedModel(nn.Module):
    def __init__(self, num_classes=3, input_size=85, hidden_size=128):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        
        # Unexpected key "attention" 대응 (Sequential 인덱스 0, 2)
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64), # 0
            nn.Tanh(),                      # 1
            nn.Linear(64, 1)                # 2
        )
        
        # Unexpected key "classifier" 대응 (Sequential 인덱스 1, 4, 7)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),                # 0
            nn.Linear(hidden_size * 2, 128), # 1
            nn.ReLU(),                      # 2
            nn.Dropout(0.5),                # 3
            nn.Linear(128, 64),              # 4
            nn.ReLU(),                      # 5
            nn.Dropout(0.5),                # 6
            nn.Linear(64, num_classes)       # 7
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.classifier(context)

# 2. 설정
MODEL_PATH = '/home/ubuntu/dev_ws/mldl/v2/checkpoints/best_model.pth'
CLASSES = ['fall', 'fight', 'normal']
COLORS = {'fall': (0, 0, 255), 'fight': (0, 165, 255), 'normal': (0, 255, 0)}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LENGTH = 30

def preprocess(kpts_list):
    sequence = np.array(kpts_list)
    nose = sequence[:, 0:1, :2].copy()
    sequence[:, :, :2] -= nose
    v = np.zeros_like(sequence[:, :, :2])
    v[1:] = sequence[1:, :, :2] - sequence[:-1, :, :2]
    combined = np.concatenate([sequence, v], axis=-1).reshape(SEQ_LENGTH, -1)
    return torch.FloatTensor(combined).unsqueeze(0).to(DEVICE)

def main():
    model = FinalCorrectedModel().to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    
    # 가중치 로드
    model.load_state_dict(state_dict)
    model.eval()

    yolo = YOLO('yolo11n-pose.pt')
    cap = cv2.VideoCapture(0)
    buffer = deque(maxlen=SEQ_LENGTH)
    
    label, conf_val = "Scanning", 0.0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        results = yolo(frame, verbose=False)
        img = frame.copy()

        for r in results:
            if r.keypoints is not None and len(r.keypoints.data) > 0:
                kpts = r.keypoints.data[0].cpu().numpy()
                buffer.append(kpts)

                if len(buffer) == SEQ_LENGTH:
                    with torch.no_grad():
                        out = model(preprocess(list(buffer)))
                        prob = F.softmax(out, dim=1)
                        c, i = torch.max(prob, dim=1)
                        label, conf_val = CLASSES[i.item()], c.item() * 100

                if r.boxes is not None:
                    box = r.boxes.xyxy[0].cpu().numpy()
                    x1, y1, x2, y2 = map(int, box)
                    color = COLORS.get(label, (255, 255, 255))
                    
                    # 시각화 (BBox + 라벨% + RGB)
                    txt = f"{label.upper()} {conf_val:.1f}%"
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    t_size = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
                    cv2.rectangle(img, (x1, y1 - t_size[1] - 15), (x1 + t_size[0] + 10, y1), color, -1)
                    cv2.putText(img, txt, (x1 + 5, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        cv2.imshow('Test', img)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

/tmp/ipykernel_3548060/2387186230.py:67: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
